In [29]:
import sys
sys.path.append('../..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [30]:
date = read_table("select * from sc_gold.dim_date")
sector= read_table("select * from sc_gold.dim_sector")

In [31]:
df = read_table(
    """
    select 
    a.year,
    a.sector,
    a.total_emp_graduate,
    b.mean_salary_rm
    from sc_bronze.dosm_emp_graduates_sector a
    left join sc_bronze.dosm_mean_salary_sector b
    on a.year = b.year and a.sector = b.sector  
    """
)
df["date"] = pd.to_datetime(df["year"], format="%Y")
df.head()

,year,sector,total_emp_graduate,mean_salary_rm,date
0,2016,Agriculture,42500.0,3374.0,2016-01-01
1,2016,Construction,202500.0,4162.0,2016-01-01
2,2016,Manufacturing,424800.0,3935.0,2016-01-01
3,2016,Mining and Quarrying,48600.0,6777.0,2016-01-01
4,2016,Services,2758500.0,4363.0,2016-01-01


In [32]:
df = df.merge(
    date[["date", "date_id"]],
    on="date",
    how="left"
)

df = df.merge(
    sector[["sector", "sector_id"]],
    on="sector",
    how="left"
)

df_final = df.drop(columns=["year", "date", "sector"])
id_cols = ["date_id", "sector_id"]
df_final = df_final[id_cols + [col for col in df_final.columns if col not in id_cols]]

In [33]:
df_final["dsec_id"] = ["DSEC" + str(i+1).zfill(4) for i in range(len(df_final))]
df_final = df_final[["dsec_id"] + [c for c in df_final.columns if c != "dsec_id"]]
df_final

,dsec_id,date_id,sector_id,total_emp_graduate,mean_salary_rm
0,DSEC0001,DT001,SEC001,42500.0,3374.0
1,DSEC0002,DT001,SEC002,202500.0,4162.0
2,DSEC0003,DT001,SEC003,424800.0,3935.0
3,DSEC0004,DT001,SEC004,48600.0,6777.0
4,DSEC0005,DT001,SEC005,2758500.0,4363.0
5,DSEC0006,DT005,SEC001,46200.0,3770.0
6,DSEC0007,DT005,SEC002,219800.0,4268.0
7,DSEC0008,DT005,SEC003,486800.0,4019.0
8,DSEC0009,DT005,SEC004,39900.0,9757.0
9,DSEC0010,DT005,SEC005,2887300.0,4751.0


In [34]:
write_table(df_final, "sc_gold", "fact_date_sector")

Table sc_gold.fact_date_sector written successfully.
